# Setting up the notebook and realtive paths

In [1]:
# Discover repo root and read all CSV files from the per-series folders
from pathlib import Path
import pandas as pd
import sys

# Find repo root
repo_root = Path.cwd()
for candidate in [repo_root] + list(repo_root.parents):
    if (candidate / 'pyproject.toml').exists() or (candidate / '.git').exists():
        repo_root = candidate
        break

# Add repo root to sys.path (BEFORE the import attempt)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
    print(f'Added {repo_root} to sys.path')

# Import plot function
try:
    from functions.plot_functions import plot_groundwater_with_flags
except ImportError as e:
    print('Failed to import plotting function:', e)
    print('Verify that functions/plot_functions.py exists and contains the function.')
finally:
    print('Import attempt finished.')

wiertsema_dir = repo_root / 'output_data' / 'wiertsema'
fugro_dir = repo_root / 'output_data' / 'fugro'
# Directory containing meteorological/stressor CSVs
stressor_dir = repo_root / 'input_stressors'
# Explicit stressor file paths used elsewhere in notebooks
precip_path = stressor_dir / 'knmi_berkhout_hourly_rain.csv'
evap_path = stressor_dir / 'knmi_berkhout_hourly_makkink.csv'

print('wiertsema dataset root ->', wiertsema_dir)
print('fugro dataset root    ->', fugro_dir)
print('precip_path ->', precip_path)
print('evap_path  ->', evap_path)

Added d:\Users\jvanruitenbeek\data_validation to sys.path
Import attempt finished.
wiertsema dataset root -> d:\Users\jvanruitenbeek\data_validation\output_data\wiertsema
fugro dataset root    -> d:\Users\jvanruitenbeek\data_validation\output_data\fugro
precip_path -> d:\Users\jvanruitenbeek\data_validation\input_stressors\knmi_berkhout_hourly_rain.csv
evap_path  -> d:\Users\jvanruitenbeek\data_validation\input_stressors\knmi_berkhout_hourly_makkink.csv


# Creating the boxplots for a folder

In [2]:
# # Loop over all Fugro CSV files and generate head distribution plots

# # Create output directory for plots
# boxplot_output_dir = out_fig / 'wiertsema_data_distribution'
# boxplot_output_dir.mkdir(parents=True, exist_ok=True)

# # Get all CSV files from fugro folder
# input_boxplot_csv_folder = sorted(wiertsema_dir.glob('*.csv'))          #
# print(f'Found {len(input_boxplot_csv_folder)} CSV files\n')

# # Loop over each file
# for i, csv_file in enumerate(input_boxplot_csv_folder, start=1):
#     try:
#         print(f'[{i}/{len(input_boxplot_csv_folder)}] Processing: {csv_file.name}')
        
#         # Read the CSV
#         df = pd.read_csv(
#             csv_file,
#             index_col=0,
#             parse_dates=True,
#             encoding="utf-8-sig",
#             encoding_errors="replace"
#         )
        
#         # Coerce all columns to numeric
#         for col in df.columns:
#             df[col] = pd.to_numeric(df[col], errors="coerce")
        
#         # Select the first numeric column as the head data
#         numeric_cols = df.select_dtypes(include=['number']).columns
#         if len(numeric_cols) == 0:
#             print(f'  ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã¢â‚¬Â¦ÃƒÂ¢Ã¢â€šÂ¬Ã…â€œÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â No numeric columns found\n')
#             continue
        
#         # The function expects a DataFrame with a "head" column, so rename the column
#         head_df = df[[numeric_cols[0]]].rename(columns={numeric_cols[0]: 'head'})
        
#         # Generate plot using the imported function
#         fig = plot_head_distribution(head_df, title=f'Head Distribution - {csv_file.stem}')
        
#         # Save as HTML
#         output_file = boxplot_output_dir / f'{csv_file.stem}.html'
#         fig.write_html(str(output_file))
#         print(f'  ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã¢â‚¬Â¦ÃƒÂ¢Ã¢â€šÂ¬Ã…â€œÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã¢â‚¬Å“ Saved: {output_file.name}\n')
        
#     except Exception as e:
#         print(f'  ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã¢â‚¬Â¦ÃƒÂ¢Ã¢â€šÂ¬Ã…â€œÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â Error processing {csv_file.name}: {e}\n')

# print(f'ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã¢â‚¬Â¦ÃƒÂ¢Ã¢â€šÂ¬Ã…â€œÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã¢â‚¬Å“ All plots saved to {boxplot_output_dir}')

# Creating the marked head plots for a folder 

In [3]:
# Loop over all validated CSV files and generate flagged head time series plots
# Choose dataset: 'wiertsema' or 'fugro'
dataset_choice = 'wiertsema'

if dataset_choice.lower() == 'wiertsema':
    dataset_root = wiertsema_dir
elif dataset_choice.lower() == 'fugro':
    dataset_root = fugro_dir
else:
    raise ValueError("dataset_choice must be 'wiertsema' or 'fugro'")

# Recursive input pattern: <dataset_root>/<origin>/validated/*.csv
input_timeseries_csv_folder = sorted(dataset_root.glob('*/validated/*.csv'))
print(f'Selected dataset root: {dataset_root}')
print(f'Found {len(input_timeseries_csv_folder)} CSV files\\n')

# Helper mapping for flexible column names
evap_aliases = ["Evapotranspiration", "evaporation", "ET", "Evapo"]
prec_aliases = ["Precipitation", "precipitation", "Rain", "P"]

for i, csv_file in enumerate(input_timeseries_csv_folder, start=1):
    source_origin_stem = csv_file.parent.parent.name
    print(f'[{i}/{len(input_timeseries_csv_folder)}] Processing: {csv_file.name} (origin={source_origin_stem})')

    try:
        df = pd.read_csv(
            csv_file,
            index_col=0,
            parse_dates=[0],
            date_format='mixed',
            encoding="utf-8-sig",
            encoding_errors="replace",
        ).reset_index()

        # Convert numeric columns safely
        df[df.columns[1:]] = df[df.columns[1:]].apply(pd.to_numeric, errors='coerce')

        # Auto-detect evaporation + precipitation columns
        evap_col = next((c for c in evap_aliases if c in df.columns), None)
        prec_col = next((c for c in prec_aliases if c in df.columns), None)

        print(df.info())

        # Generate plot
        fig = plot_groundwater_with_flags(
            df,
            evap_col=evap_col,
            prec_col=prec_col
        )

        # Save output in same origin folder: <origin>/figures/*.html
        output_dir = dataset_root / source_origin_stem / 'figures'
        output_dir.mkdir(parents=True, exist_ok=True)
        output_file = output_dir / f'{csv_file.stem}.html'
        fig.write_html(str(output_file))
        print(f'  [OK] Saved -> {output_file}\\n')

    except Exception as e:
        print(f'  [ERR] Error: {e}\\n')

print(f'[OK] All time series plots saved under dataset root -> {dataset_root}')

Selected dataset root: d:\Users\jvanruitenbeek\data_validation\output_data\wiertsema
Found 281 CSV files\n
[1/281] Processing: 83034-1 HB001PB01 BE0049+00_BUKR_GMW_PB1_F-229.csv (origin=83034-1)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4272 entries, 0 to 4271
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Time                4272 non-null   datetime64[ns]
 1   head                3947 non-null   float64       
 2   head_raw            3951 non-null   float64       
 3   Precipitation       117 non-null    float64       
 4   Evapotranspiration  117 non-null    float64       
 5   recharge            117 non-null    float64       
 6   v0                  4272 non-null   bool          
 7   head_raw_3d         4073 non-null   float64       
 8   head_raw_7d         4169 non-null   float64       
 9   v1                  0 non-null      float64       
 10  v2                  0

# Creating Validated CSV files